<a id="top"></a>

# Lab L1205: Pass rates and regressions

```yaml
title: "Lab L1205: Pass rates and regressions"
keywords: evaluation, pass rate, samples, non-determinism, regression, compare, model comparison, lab
estimated duration: 30
```

> **Lesson:** L12. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L12/objectives.md).
> Runs offline — no API key needed (a deterministic model simulator).

**Learning objectives.** By the end of this lab you can:

- **move from a verdict to a pass rate** — run a case K times and read how often it passes;
- **flag regressions** between two versions with `compare(...)` — pass→fail is the thing to catch.

> Solutions: `L0905_lab_solutions.ipynb`. Pure Python — no API key needed.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — a pass rate, not a verdict](#2-problem-1--a-pass-rate-not-a-verdict)
- [3. Problem 2 — flag the regressions](#3-problem-2--flag-the-regressions)
- [4. Problem 3 — read the instrument](#4-problem-3--read-the-instrument)
- [5. Self-check](#5-self-check)

## 1. Setup

Run this cell once. The agent loop is non-deterministic, so we provide a **deterministic simulator** of it: `ModelSim(strong=...)` is a `run_case` that behaves like a stronger or weaker model. A strong model recovers from a tool error and looks a city up once; a weak model gives up on recovery and *intermittently runs away* on the lookup — so its pass rate is a fraction, reproducibly. You'll run the eval set and read the numbers.

In [1]:
from fluffy_potato_curriculum.common.agent_loop import RunResult, run
from fluffy_potato_curriculum.common.evals import (
    EvalCase,
    EvalResult,
    compare,
    evaluate,
    tool_calls,
)
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    response,
    text_block,
    tool_use_block,
)
from fluffy_potato_curriculum.common.tools import TOOLS


def answer_correct(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Outcome scorer (loosened): is the reference answer in the final text?"""

    def norm(text: str) -> str:
        return text.replace(",", "").replace(" ", "")

    expected = str(example.reference_outputs["answer"])
    return EvalResult(key="answer_correct", score=norm(expected) in norm(run.final_text))


def no_runaway(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Trajectory scorer: no repeated (tool, args) call and didn't hit the step cap."""
    signatures = [(name, tuple(sorted(args.items()))) for name, args in tool_calls(run)]
    repeated = len(signatures) != len(set(signatures))
    return EvalResult(key="no_runaway", score=not (repeated or run.termination == "max_steps"))


class ModelSim:
    """A deterministic run_case simulating a model of a given strength."""

    def __init__(self, *, strong: bool) -> None:
        self.strong = strong
        self._seen: dict[str, int] = {}

    def _attempt(self, case_id: str) -> int:
        n = self._seen.get(case_id, 0)
        self._seen[case_id] = n + 1
        return n

    def __call__(self, case: EvalCase) -> RunResult:
        n = self._attempt(case.id)
        if case.id == "recover" and self.strong:
            blocks = [
                response([tool_use_block("c1", "flaky_fetch", {"url": "https://crash"})]),
                response([tool_use_block("c2", "flaky_fetch", {"url": "https://ok"})]),
                response([text_block("The answer is 42.")]),
            ]
        elif case.id == "recover":
            blocks = [
                response([tool_use_block("c1", "flaky_fetch", {"url": "https://crash"})]),
                response([text_block("Sorry, I could not fetch that URL.")]),
            ]
        elif case.id == "hard_lookup" and not self.strong and n % 3 == 1:
            blocks = [response([tool_use_block("c1", "lookup", {"city": "Atlantis"})])]  # runaway
        elif case.id == "hard_lookup":
            blocks = [
                response([tool_use_block("c1", "lookup", {"city": "Tokyo"})]),
                response([text_block("Tokyo has a population of 37000000.")]),
            ]
        else:
            raise KeyError(case.id)
        return run(
            FakeModel(blocks), TOOLS, case.inputs["task"], max_steps=case.inputs.get("max_steps", 8)
        )


recover_case = EvalCase(
    id="recover",
    inputs={"task": "Fetch from https://crash; if it fails try https://ok."},
    reference_outputs={"answer": "42"},
)
hard_lookup_case = EvalCase(
    id="hard_lookup",
    inputs={"task": "What is the population of Tokyo?", "max_steps": 4},
    reference_outputs={"answer": "37000000"},
)
cases = [recover_case, hard_lookup_case]
scorers = [answer_correct, no_runaway]
print("setup ready:", [c.id for c in cases])

setup ready: ['recover', 'hard_lookup']


[↑ Back to top](#top)

## 2. Problem 1 — a pass rate, not a verdict

A single run can be luck. Run the **weak** model on the lookup case `samples=5` times and read the pass *rate* for the `no_runaway` check. Because the weak model intermittently runs away, the rate is a **fraction** — that fraction is the finding.

Fill in the `evaluate(...)` call (set `samples=5`) and read the rate with `report.pass_rate(case_id, key)`.

In [2]:
weak = ModelSim(strong=False)
report = evaluate(weak, [hard_lookup_case], scorers, samples=5)
print(report.table())

rate = report.pass_rate("hard_lookup", "no_runaway")
print("no_runaway pass rate:", rate)

# A flaky case: passes sometimes, fails sometimes -> strictly between 0 and 1.
assert 0.0 < rate < 1.0

case         answer_correct  no_runaway  ALL
hard_lookup  3/5             3/5         3/5
no_runaway pass rate: 0.6


[↑ Back to top](#top)

## 3. Problem 2 — flag the regressions

Now use the eval set as a **ratchet**. Build a report for the strong model (the baseline) and one for the weak model (the "after"), then call `compare(before, after)` to flag which `(case, scorer)` pairs went **pass → fail**.

Fill in the two `evaluate(...)` calls and the `compare(...)` call. The weak model should regress recovery and the lookup.

In [3]:
strong_report = evaluate(ModelSim(strong=True), cases, scorers, samples=5)
weak_report = evaluate(ModelSim(strong=False), cases, scorers, samples=5)

delta = compare(strong_report, weak_report)
print("regressions (pass -> fail):", delta.regressions)
print("fixes       (fail -> pass):", delta.fixes)

# Recovery should be a clear regression for the weaker model.
assert ("recover", "answer_correct") in delta.regressions
assert delta.has_regressions

regressions (pass -> fail): [('recover', 'answer_correct'), ('hard_lookup', 'answer_correct'), ('hard_lookup', 'no_runaway')]
fixes       (fail -> pass): []


[↑ Back to top](#top)

## 4. Problem 3 — read the instrument

The eval set is a **measurement instrument**, not just a guard. Print the strong and weak tables side by side, then answer the written question.

In [4]:
print("=== strong model ===")
print(evaluate(ModelSim(strong=True), cases, scorers, samples=5).table())
print("\n=== weak model ===")
print(evaluate(ModelSim(strong=False), cases, scorers, samples=5).table())

=== strong model ===
case         answer_correct  no_runaway  ALL
recover      5/5             5/5         5/5
hard_lookup  5/5             5/5         5/5

=== weak model ===
case         answer_correct  no_runaway  ALL
recover      0/5             5/5         0/5
hard_lookup  3/5             3/5         3/5


**TODO (written):** A teammate runs each case **once** and reports "the weak model passed recovery." Using what you saw above, explain in one or two sentences why that single run can't be trusted, and what number you'd report instead. Edit this cell.

_TODO_

[↑ Back to top](#top)

## 5. Self-check

Compare against `L0905_lab_solutions.ipynb`. Done when:

- **Problem 1** the `no_runaway` pass rate for `hard_lookup` is strictly between 0 and 1 (a fraction like `3/5`), and the assert passes.
- **Problem 2** `compare(strong, weak)` flags `("recover", "answer_correct")` (and the lookup checks) as regressions; both asserts pass.
- **Problem 3** your written answer explains that one run is a coin flip on a non-deterministic agent, and that you'd report a **pass rate over K samples** instead.

You can now measure pass rates and flag regressions across two versions — the ratchet you'll carry into every later agent. Next: [`L0906_lecture`](L0906_lecture.ipynb) asks what an eval run *costs*.

[↑ Back to top](#top)